# 货币利润TTM标签覆盖修复
对实际观察到的us-gaap:ProfitLoss扩展解析，作为独立合并利润口径，绝不与NetIncomeLoss跨标签拼接，亦不称普通股归属利润。其余冻结规则及原始事实不变，没有价格、收益或模型。

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'SPEC.md').exists(): root=root.parent
sys.path.insert(0,str(root))
import qlib
from qlib.data import D
qlib.init(provider_uri=str(root/'data/qlib/soxx_pit_leveraged_training_v1'),region='us',kernels=1)
calendar=D.calendar(start_time='2019-01-01',end_time='2026-07-02')
from src.data.etf_valuation_ttm import run
summary=run(root,card_name='etf_pit_ttm_tag_coverage_20261016')
summary

[77585:MainThread](2026-10-16 11:58:15,631) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/16 11:58:15 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[77585:MainThread](2026-10-16 11:58:16,004) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[77585:MainThread](2026-10-16 11:58:16,005) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[77585:MainThread](2026-10-16 11:58:16,765) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[77585:MainThread](2026-10-16 11:58:16,765) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[77585:MainThread](2026-10-16 11:58:16,766) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1')}


{'experiment': 'etf_pit_ttm_tag_coverage_20261016',
 'decision': 'candidate_monetary_TTM_constructed_not_EP_qualified',
 'objective_achieved': False,
 'new_http_requests': 0,
 'new_fits': 0,
 'new_backtests': 0,
 'coverage_rows': 118,
 'issuer_detail_rows': 3210,
 'monthly_dates': 90,
 'months_with_public_holdings': 80,
 'monthly_candidate95pct': 23,
 'disclosures_candidate95pct': 10,
 'monthly_min_candidate_weight_pct': 85.911316356679,
 'monthly_max_candidate_weight_pct': 99.855605320658,
 'monthly_first': '2019-01-31',
 'monthly_last': '2026-06-30',
 'qualified_ep_monthly_rows': 0,
 'qualified_input_path': None,
 'numerator_scope_compatible_certified': False,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/soxx_pit_leveraged_training_v1',
 'region': 'us',
 'market': 'SOXX historical basket',
 'as_of': '2026-07-02',
 'calendar_last': '2026-07-02 00:00:00',
 'source_binding': {'source_card': 'research/etf_pit_fundamentals_repair_20261016',
  'source_summary_sha256': 'c3f26297

In [2]:
from qlib.workflow import R
R.set_uri(summary['recorder_uri'])
rec=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
rec.load_object('analysis/coverage')

,kind,cutoff,basket_id,holdings_rows,equity_weight_pct,candidate_ttm_weight_pct,known_negative_candidate_weight_pct,annual_direct_weight_pct,derived_ttm_weight_pct,missing_income_symbols,candidate95pct,qualified_ep_weight_pct
0,disclosure,2019-11-26,0001752724-19-177706,30,99.805524,92.630798,3.581838,13.451293,79.179505,ASML;TSM,False,0
1,disclosure,2020-02-28,0001752724-20-038669,30,99.751505,95.689546,3.160602,45.512480,50.177066,TSM,True,0
2,disclosure,2020-06-02,0001752724-20-112013,30,99.695402,99.695402,0.651355,15.735508,83.959893,,True,0
3,disclosure,2020-08-28,0001752724-20-176975,30,99.724723,91.903439,5.991674,11.178211,80.725228,ASML;TSM,False,0
4,disclosure,2020-11-27,0001752724-20-247778,30,99.858732,92.263557,5.566392,20.005331,72.258226,ASML;TSM,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...
113,monthly,2026-02-27,0002071691-26-004253,30,99.876847,94.585812,6.947917,49.671094,44.914718,ASX;TSM;UMC,False,0
114,monthly,2026-03-31,0002071691-26-004253,30,99.876847,94.585812,6.947917,50.491666,44.094146,ASX;TSM;UMC,False,0
115,monthly,2026-04-30,0002071691-26-004253,30,99.876847,95.447239,6.947917,38.395717,57.051522,TSM;UMC,True,0
116,monthly,2026-05-29,0002071691-26-012504,30,99.860495,95.762496,4.128442,9.971833,85.790664,TSM;UMC,True,0


In [3]:
# Provisional upper bound only; EP is still unqualified.
import pandas as pd
coverage = rec.load_object('analysis/coverage')
monthly = coverage.loc[coverage.kind.eq('monthly')].copy()
monthly['date'] = pd.to_datetime(monthly.cutoff)
monthly['prior60_candidate_count'] = [int(monthly.loc[(monthly.date < cutoff) & (monthly.date >= cutoff-pd.DateOffset(months=60)), 'candidate95pct'].sum()) for cutoff in monthly.date]
rank_feasibility = monthly[['cutoff','candidate95pct','prior60_candidate_count']]
rank_feasibility.to_csv(root/'research/etf_pit_ttm_tag_coverage_20261016/provisional_rank_feasibility.csv',index=False)
{'candidate_months_95pct': int(monthly.candidate95pct.sum()), 'maximum_prior60_candidate_count': int(monthly.prior60_candidate_count.max()), 'required_prior_distinct_months':36, 'qualified_EP_rows':0}


{'candidate_months_95pct': 23,
 'maximum_prior60_candidate_count': 18,
 'required_prior_distinct_months': 36,
 'qualified_EP_rows': 0}

# TTM标签覆盖修复结论

对真实已有的us-gaap:ProfitLoss增加独立解析，保留与NetIncomeLoss、ifrs-full:ProfitLoss的不同合并利润口径；年度和YTD从不跨标签拼接。没有改变成本、历史窗口或策略选择，也没有观察收益结果。

90个月末中80个月有已公开持仓；候选货币TTM覆盖85.91%–99.86%，23个月达到95%，10/28披露截止日达到95%。相比首版的2个月，差别来自实际财务标签覆盖修复，而非交易机制改善。所有证券明细保留taxonomy/tag/unit和每个利润组分的accession及可用时间。

即便把这23个月都暂当独立候选，任一截止日之前60个月的最大合格候选数只有18，达不到36个先验观察的分位门槛。真实EP仍没有完成股份、原始财务口径、ADR、FX、价格/拆股和篮子定义资格，合格EP月度行数0，没有拟合或回测，不晋升。不得把候选TTM覆盖当成可用ETF估值历史。

earnings_change未生成；跨NetIncomeLoss/ProfitLoss或币种变化必须未知，不能直接连成盈利增长。ETF层面的method_id还必须包含归属口径、汇率、ADR、权重版本、负盈利和缺失处理，见feature_boundary.json。负利润保留，不把未知亏损填零。

剩余来源缺口以外国发行人的季度资料为主，而非完整公司事实完全不可得。后续已启动固定6URL的TSM/UMC历史季度可得性检查；只判断可获取与待抽取，不能凭两个样本认证整个历史。当前工作停在这一批准范围内，稳定超额目标仍未实现，原始记录和前版结果均保留。
